# Experiment: can the LiDAR archives be unpacked on all cores? (CPU, high RAM)

Unpacking the compressed LiDAR layer was the largest single wait in a block. The dataset toolkit unpacks each file on
ONE core. The files were compressed with 64 threads, which usually means they consist of many independent pieces that
can be unpacked side by side. This notebook measures whether that works here, and checks that the unpacked files
are byte-for-byte the same. Nothing in the project is changed by this test.

**Outcome: 5.3 times faster (419 s to 79 s), every file byte-identical.** Built into `src/vggt_aura/fast_download.py`,
with a checksum before unpacking and an automatic fall-back to the toolkit's own way.

In [1]:
# --- 1. Configuration ---
PERSIST_MODE = "drive"
SPLIT, BLOCK = "val", 11                  # one LiDAR file of 3.8 GB: the known case
LAYER = "lidar_motion_compensated_keyframes"

In [ ]:
# === CODE SYNC (auto-generated by `python -m vggt_aura.sync`, do not edit) ===
raise RuntimeError("The sync cell is empty. On your own machine, in the project folder, run:  python -m vggt_aura.sync   and reopen this notebook.")

In [3]:
# --- 3. Session, cores, and the unpacking tool ---
import os, subprocess
from vggt_aura.session import start_session

session = start_session(persist_mode=PERSIST_MODE, require_gpu=False)
print("CPU cores on this server:", os.cpu_count())
version = subprocess.run("xz --version", shell=True, capture_output=True, text=True)
print(version.stdout.strip() or "xz is NOT installed: this test cannot run here")
print("(parallel unpacking needs xz 5.4 or newer)")

Mounted at /content/drive
installing vggt_omega
installing pybind11
installing fzi_aura
persist root: /content/drive/MyDrive/vggt-omega-aura-benchmark
data root   : /content/data/fzi-aura (runtime disk, wiped at session end)
CPU cores on this server: 8
xz (XZ Utils) 5.4.5
liblzma 5.4.5
(parallel unpacking needs xz 5.4 or newer)


In [4]:
# --- 4. Fetch the compressed LiDAR file(s) of the block, without unpacking ---
import time
from pathlib import Path
from huggingface_hub import hf_hub_download
from vggt_aura import aura_data as ad, pins

chunks, scene_blocks, hub_files = ad.fetch_release_tables(session.data_root / "_release_tables")
wanted = chunks[(chunks["split"] == SPLIT) & (chunks["scene_block"] == BLOCK) & (chunks["layer"] == LAYER)]
work = session.data_root / "_unpack_test"
work.mkdir(parents=True, exist_ok=True)
started = time.time()
archives = [Path(hf_hub_download(pins.AURA_DATASET_REPO, path, repo_type="dataset", revision=pins.AURA_DATASET_REVISION,
                                 local_dir=str(work / "archives"))) for path in wanted["chunk_path"]]
print(f"{len(archives)} file(s), {sum(a.stat().st_size for a in archives) / 1e9:.2f} GB, fetched in {time.time() - started:.0f} s")
for archive in archives:
    listing = subprocess.run(["xz", "--robot", "--list", str(archive)], capture_output=True, text=True).stdout
    totals = [line.split("\t") for line in listing.splitlines() if line.startswith("totals")]
    print(f"  {archive.name}: independent pieces inside = {totals[0][2] if totals else 'unknown'}"
          "  (1 means it cannot be unpacked in parallel)")

1 file(s), 3.84 GB, fetched in 64 s
  val-lidar-motion-compensated-keyframes-block-000011-part-000000.tar.xz: independent pieces inside = 324  (1 means it cannot be unpacked in parallel)


In [5]:
# --- 5. Unpack three ways, time each, fingerprint the result, delete it again (keeps the disk small) ---
import hashlib, shutil, tarfile


def fingerprint(root):
    found = {}
    for item in sorted(root.rglob("*")):
        if item.is_file():
            found[item.relative_to(root).as_posix()] = (item.stat().st_size, hashlib.sha256(item.read_bytes()).hexdigest())
    return found


def python_way(target):
    for archive in archives:
        with tarfile.open(archive, "r:xz") as tar:
            tar.extractall(target)


def xz_way(threads):
    def run(target):
        for archive in archives:
            subprocess.run(f'xz -T{threads} -dc "{archive}" | tar -x -C "{target}"', shell=True, check=True)
    return run


def trial(label, unpack):
    target = work / "out"
    shutil.rmtree(target, ignore_errors=True)
    target.mkdir(parents=True)
    started = time.time()
    unpack(target)
    seconds = time.time() - started
    print(f"{label:34} {seconds:7.1f} s")
    found = fingerprint(target)
    shutil.rmtree(target, ignore_errors=True)
    return seconds, found


results = {"python": trial("Python, as the toolkit does", python_way),
           "one_core": trial("xz tool, one core", xz_way(1)),
           "all_cores": trial(f"xz tool, all {os.cpu_count()} cores", xz_way(0))}

/tmp/ipykernel_3651/4251052603.py:16: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(target)


Python, as the toolkit does          419.2 s
xz tool, one core                    417.3 s
xz tool, all 8 cores                  79.3 s


In [6]:
# --- 6. Verdict: identical files, and how much faster ---
reference = results["python"][1]
identical = all(found == reference for _, found in results.values())
print(f"files: {len(reference)} | total {sum(size for size, _ in reference.values()) / 1e9:.2f} GB | all three ways give identical files: {identical}")
gain = results["python"][0] / results["all_cores"][0]
print(f"toolkit's way {results['python'][0] / 60:.1f} min  vs  all cores {results['all_cores'][0] / 60:.1f} min  ->  {gain:.1f}x")
if not identical:
    print("VERDICT: the outputs DIFFER. Do not use parallel unpacking, and report it.")
elif gain >= 2.0:
    print("VERDICT: WORTH BUILDING IN. Identical files, clearly faster.")
else:
    print("VERDICT: not worth it. Identical files, but too small a gain to justify replacing the toolkit's unpacking.")
shutil.rmtree(work, ignore_errors=True)

files: 4798 | total 8.13 GB | all three ways give identical files: True
toolkit's way 7.0 min  vs  all cores 1.3 min  ->  5.3x
VERDICT: WORTH BUILDING IN. Identical files, clearly faster.
